## Imports

In [ ]:
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Settings

In [ ]:
# Visual style settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

APPROACHES_CONFIG = [
    {'det': 'yolov8n', 'trk': 'bytetrack', 'name': 'YOLOv8n + ByteTrack'},
    {'det': 'yolov8n', 'trk': 'botsort', 'name': 'YOLOv8n + BoT-SORT'},
    {'det': 'yolov8s', 'trk': 'bytetrack', 'name': 'YOLOv8s + ByteTrack'},
    {'det': 'yolov8s', 'trk': 'botsort', 'name': 'YOLOv8s + BoT-SORT'},
]

## Paths resolving and preparation

In [ ]:
# Select input directories explicitly
SPLIT = os.environ.get("SPLIT", "dev").strip().lower()
if SPLIT not in {"dev", "test"}:
    raise ValueError(f"Unsupported SPLIT={SPLIT!r}")


def find_root(start):
    for directory in (start, *start.parents):
        if (directory / "src").is_dir() and (directory / "reports").is_dir():
            return directory
    raise FileNotFoundError(f"Repository root not found above {start}")


ROOT_DIR = find_root(Path.cwd().resolve())


def input_path(variable, default):
    value = Path(os.environ.get(variable, default))
    return value.resolve() if value.is_absolute() else (ROOT_DIR / value).resolve()


reports_dir = input_path("REPORTS_DIR", "reports")
detector_reports_dir = input_path("DETECTOR_REPORTS_DIR", str(reports_dir))
tracks_dir = input_path("TRACKS_ROOT", "outputs/tracks") / SPLIT
output_dir = input_path("RESULT_OUTPUT_ROOT", "outputs/reports") / SPLIT
charts_dir, tables_dir = output_dir / "charts", output_dir / "tables"
charts_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from src.utils import file_info

RUN_CONF = float(os.environ.get("RUN_CONF", "0.25"))
print(f"Split {SPLIT}\nReports {reports_dir}\nTracks {tracks_dir}\nOutput {output_dir}")


## Directory checks

In [ ]:
if not reports_dir.exists():
  raise FileNotFoundError(
      f'Detector reports directory "{reports_dir}" not found in the project root, run the evaluation scripts'
  )

if not tracks_dir.exists():
  raise FileNotFoundError(
      f'Tracks directory "{tracks_dir}" not found, run scripts/run_tracking.py'
  )

## Tables preparation

In [ ]:
rows = []
class_rows = []
detector_data = {}
print(f'Collecting metrics for 4 approaches... Split: {SPLIT.upper()}')


def percent(value):
    """Convert a 0..1 metric to percent; preserve missing values."""
    return round(float(value) * 100, 2) if value is not None else None


def macro_percent(values):
    values = [float(v) for v in values if v is not None and np.isfinite(float(v))]
    return round(float(np.mean(values)) * 100, 2) if values else None


# Read one detector report per model. Detection quality is detector level, not tracker level
for detector in ('yolov8n', 'yolov8s'):
    det_json_path = detector_reports_dir / f'detection_{detector}_{SPLIT}.json'
    if not det_json_path.exists():
        raise FileNotFoundError(
            f'Detector report not found: {det_json_path}. Run scripts/evaluate_detectors.py first.'
        )

    det_data = json.loads(det_json_path.read_text(encoding='utf-8'))
    if det_data.get("split") != SPLIT:
        raise ValueError(f"Detector split mismatch in {det_json_path}")
    detector_data[detector] = det_data

    for class_name, metrics in det_data.get('per_class', {}).items():
        if not isinstance(metrics, dict):
            continue
        class_rows.append({
            'Detector': detector,
            'Class': class_name,
            'TP': metrics.get('TP'),
            'FP': metrics.get('FP'),
            'FN': metrics.get('FN'),
            'Precision@0.25 (%)': percent(metrics.get('P@0.25')),
            'Recall@0.25 (%)': percent(metrics.get('R@0.25')),
            'F1@0.25 (%)': percent(metrics.get('F1@0.25')),
            'AP50_95 (%)': percent(metrics.get('AP50_95')),
        })

for item in APPROACHES_CONFIG:
    det_name = item['det']
    trk_name = item['trk']
    display_name = item['name']
    det_data = detector_data[det_name]

    config = f"{det_name}_{trk_name}_conf{RUN_CONF:g}"
    target_track_dir = tracks_dir / config
    timing_file = target_track_dir / "timing.json"
    timing_data = json.loads(timing_file.read_text(encoding="utf-8"))
    if timing_data["split"] != SPLIT or timing_data["imgsz"] != det_data["imgsz"]:
        raise ValueError(f"Report settings differ for {display_name}")
    if timing_data["tracker"] != trk_name or timing_data["conf"] != RUN_CONF:
        raise ValueError(f"Tracking settings differ for {display_name}")
    if Path(timing_data["detector"]).stem != det_name:
        raise ValueError(f"Detector name mismatch for {display_name}")
    item["config"] = config
    item["timing_source"] = file_info(timing_file)

    per_class = det_data.get('per_class', {})
    precisions = [m.get('P@0.25') for m in per_class.values() if isinstance(m, dict)]
    recalls = [m.get('R@0.25') for m in per_class.values() if isinstance(m, dict)]
    f1_scores = [m.get('F1@0.25') for m in per_class.values() if isinstance(m, dict)]

    rows.append({
        'Approach': display_name,
        'Detector': det_name,
        'Tracker': trk_name,
        'mAP_50 (%)': percent(det_data.get('mAP@0.5')),
        'mAP_50_95 (%)': percent(det_data.get('mAP@0.5:0.95')),
        'mAP_75 (%)': percent(det_data.get('mAP@0.75')),
        'Precision_macro (%)': macro_percent(precisions),
        'Recall_macro (%)': macro_percent(recalls),
        'F1_macro (%)': macro_percent(f1_scores),
        'FPS': round(float(timing_data['fps']), 2),
        'Latency_ms': round(float(timing_data['total_ms_per_frame']), 2),
    })

df = pd.DataFrame(rows)
class_df = pd.DataFrame(class_rows)

print('\nSummary table: four detector-tracker configurations')
print(df.to_markdown(index=False))

print('\nPer-class detection table')
if class_df.empty:
    print('No per-class metrics found. Check that detector evaluation was rerun.')
else:
    print(class_df.to_markdown(index=False))


## Tracking tables (IDF1, IDSW, HOTA, MOTA)

In [ ]:
tracking_json = reports_dir / f'tracking_eval_{SPLIT}.json'
if not tracking_json.exists():
    raise FileNotFoundError(
        f'Tracking eval not found: {tracking_json}. '
        'Run `python scripts/evaluate_tracking.py --splits {SPLIT}` first.'
    )
tracking_data = json.loads(tracking_json.read_text(encoding='utf-8'))

track_rows = []
expected_configs = {item["config"] for item in APPROACHES_CONFIG}
if set(tracking_data) != expected_configs:
    raise ValueError("Tracking report configurations differ from selected runs")
for item in APPROACHES_CONFIG:
    cfg = item["config"]
    body = tracking_data[cfg]
    recorded_timing = body.get("sources", {}).get("timing")
    if recorded_timing and recorded_timing["sha256"] != item["timing_source"]["sha256"]:
        raise ValueError(f"Tracking report refers to a different run for {cfg}")
    agg = body['aggregate']
    metrics = agg['combined']
    track_rows.append({
        'Approach': cfg.replace('_conf0.25', '').replace('_', ' ').title(),
        'Config': cfg,
        'HOTA (%)': metrics['HOTA'],
        'DetA (%)': metrics['DetA'],
        'AssA (%)': metrics['AssA'],
        'MOTA (%)': metrics['MOTA'],
        'MOTP (%)': metrics['MOTP'],
        'IDF1 (%)': metrics['IDF1'],
        'IDSW': agg['sum']['IDSW'],
        'MT': agg['sum']['MT'],
        'ML': agg['sum']['ML'],
    })
track_df = pd.DataFrame(track_rows)
print('Tracking metrics combined across the dataset')
print(track_df.to_markdown(index=False))

## Plots

In [ ]:
# Figure 1: detector quality (one bar per detector) and pipeline speed quality trade off
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
detector_df = df.drop_duplicates('Detector').sort_values('Detector')
map_cols = ['mAP_50 (%)', 'mAP_50_95 (%)']
detector_melted = detector_df.melt(
    id_vars='Detector', value_vars=map_cols, var_name='Metric', value_name='Score (%)'
)
sns.barplot(data=detector_melted, x='Detector', y='Score (%)', hue='Metric', ax=axes[0])
axes[0].set_title('Detector quality', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Score (%)')
axes[0].set_ylim(0, 100)
axes[0].legend(title='Metric')

sns.scatterplot(
    data=df, x='FPS', y='mAP_50 (%)', hue='Approach', style='Approach', s=180, ax=axes[1]
)
axes[1].set_title('Detection quality vs pipeline speed', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Speed (FPS)')
axes[1].set_ylabel('mAP@0.5 (%)')
for _, row in df.iterrows():
    axes[1].annotate(
        f"{row['Approach']}\n({row['Latency_ms']} ms)",
        (row['FPS'], row['mAP_50 (%)']),
        xytext=(5, 5), textcoords='offset points', fontsize=7
    )
plt.tight_layout()
summary_plot = charts_dir / f'models_comparison_charts_{SPLIT}.png'
plt.savefig(summary_plot, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {summary_plot}')

# Figure 2: per class Precision, Recall and F1, separately for each detector
if not class_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    prf_cols = ['Precision@0.25 (%)', 'Recall@0.25 (%)', 'F1@0.25 (%)']
    for ax, detector in zip(axes, ('yolov8n', 'yolov8s')):
        part = class_df[class_df['Detector'] == detector]
        plot_data = part.melt(
            id_vars='Class', value_vars=prf_cols, var_name='Metric', value_name='Score (%)'
        )
        sns.barplot(data=plot_data, x='Class', y='Score (%)', hue='Metric', ax=ax)
        ax.set_title(f'{detector}: per-class P/R/F1')
        ax.set_xlabel('Class')
        ax.set_ylabel('Score (%)')
        ax.set_ylim(0, 100)
        ax.tick_params(axis='x', rotation=15)
        ax.legend(title='Metric', fontsize=8)
    plt.tight_layout()
    prf_plot = charts_dir / f'detector_class_prf1_{SPLIT}.png'
    plt.savefig(prf_plot, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {prf_plot}')

    # Figure 3: per-class AP averaged over IoU thresholds 0.50:0.95.
    ap_data = class_df.dropna(subset=['AP50_95 (%)'])
    if not ap_data.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        sns.barplot(data=ap_data, x='Class', y='AP50_95 (%)', hue='Detector', ax=ax)
        ax.set_title('Per-class AP@0.5:0.95')
        ax.set_ylabel('AP (%)')
        ax.set_xlabel('Class')
        ax.set_ylim(0, 100)
        ax.legend(title='Detector')
        plt.tight_layout()
        ap_plot = charts_dir / f'detector_class_ap_{SPLIT}.png'
        plt.savefig(ap_plot, dpi=300, bbox_inches='tight')
        plt.show()
        print(f'Saved: {ap_plot}')


## Tracking plots (IDF1, IDSW, HOTA, MOTA)

In [ ]:
# Plot tracking quality and identity switches
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# HOTA / MOTA / IDF1 bar charts
for ax, metric in zip(axes[:3], ['HOTA (%)', 'MOTA (%)', 'IDF1 (%)']):
    sns.barplot(data=track_df, x='Approach', y=metric, ax=ax,
                palette='viridis' if metric == 'HOTA (%)' else
                        ('magma' if metric == 'MOTA (%)' else 'plasma'))
    ax.set_title(f'{metric}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)')
    ax.set_ylim(min(0, float(track_df[metric].min()) * 1.1), 100)
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}',
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=8)

# IDSW bar chart (lower is better)
ax = axes[3]
sns.barplot(data=track_df, x='Approach', y='IDSW', ax=ax, palette='Reds_r')
ax.set_title('IDSW (lower is better)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of switches')
ax.tick_params(axis='x', rotation=30)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
tracking_plot = charts_dir / f'tracking_metrics_{SPLIT}.png'
plt.savefig(tracking_plot, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {tracking_plot}')

# DetA vs AssA (scatter)
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=track_df, x='DetA (%)', y='AssA (%)',
                hue='Approach', style='Approach', s=200, ax=ax)
ax.set_title('HOTA decomposition: detection (DetA) vs association (AssA)', 
             fontsize=12, fontweight='bold')
ax.set_xlabel('DetA (%) — detection quality')
ax.set_ylabel('AssA (%) — association quality')
import numpy as np
det_grid = np.linspace(0.1, 100, 100)
for hota_val in [20, 30, 40, 50, 60, 70]:
    assa_grid = (hota_val ** 2) / det_grid
    mask = (assa_grid > 0) & (assa_grid <= 100)
    ax.plot(det_grid[mask], assa_grid[mask], '--', color='gray', alpha=0.4, lw=0.8)
    idx = np.where(mask)[0][len(np.where(mask)[0])//2]
    ax.annotate(f'sqrt(DetA * AssA)={hota_val}', (det_grid[idx], assa_grid[idx]),
                fontsize=7, color='gray', alpha=0.7)
for _, row in track_df.iterrows():
    ax.annotate(row['Approach'], (row['DetA (%)'], row['AssA (%)']),
                xytext=(8, 8), textcoords='offset points', fontsize=8)
plt.tight_layout()
hota_decomp_plot = charts_dir / f'hota_decomposition_{SPLIT}.png'
plt.savefig(hota_decomp_plot, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {hota_decomp_plot}')

## Saving the results

In [ ]:
# Save the summary comparison table (CSV, Markdown and LaTeX)
summary_csv = tables_dir / f'metrics_table_{SPLIT}.csv'
summary_md = tables_dir / f'metrics_table_{SPLIT}.md'
summary_tex = tables_dir / f'metrics_table_{SPLIT}.tex'
df.to_csv(summary_csv, index=False)
df.to_markdown(summary_md, index=False)
summary_tex.write_text(
    df.to_latex(
        index=False,
        float_format='%.2f',
        caption='Comparison of detector quality and pipeline speed for four configurations',
        label=f'tab:models_comparison_{SPLIT}',
    ),
    encoding='utf-8',
)

# Save the detailed per class detector table, including the new TP/FP/FN/F1 fields
class_csv = tables_dir / f'detector_per_class_{SPLIT}.csv'
class_md = tables_dir / f'detector_per_class_{SPLIT}.md'
class_df.to_csv(class_csv, index=False)
class_df.to_markdown(class_md, index=False)

print('Saved tables:')
for path in (summary_csv, summary_md, summary_tex, class_csv, class_md):
    print(f' - {path}')

# Save tracking metrics table (CSV, Markdown and LaTeX)
track_csv = tables_dir / f'tracking_metrics_{SPLIT}.csv'
track_md = tables_dir / f'tracking_metrics_{SPLIT}.md'
track_tex = tables_dir / f'tracking_metrics_{SPLIT}.tex'
track_df.to_csv(track_csv, index=False)
track_df.to_markdown(track_md, index=False)
track_tex.write_text(
    track_df.to_latex(
        index=False, float_format='%.2f',
        caption=f'Tracking metrics (HOTA, MOTA, IDF1, IDSW) for four configurations on {SPLIT}',
        label=f'tab:tracking_metrics_{SPLIT}',
    ),
    encoding='utf-8',
)
print(f'Saved tracking tables: {track_csv}, {track_md}, {track_tex}')